## 1.1 데이터 수집 및 방법

- **과거 데이터 (1950~2024)**  
  Kaggle의 F1 Dataset(CSV)을 활용해 기본 데이터 구축함.

- **최근 데이터 (2024~2026)**  
  - F1 공식 홈페이지를 크롤링하여 최신 경기 결과 수집함.  
  - Open API를 활용해 시즌별 우승 횟수 및 선수 통계 보완함.

---

## 1.2 수집 항목

- **드라이버 정보**  
  이름, 국적, 시즌 순위, 포인트, 생년월일  

- **팀 정보**  
  팀명, 시즌 포인트, 우승 횟수  

- **레이스 정보**  
  그랑프리명, 개최 국가/도시, 서킷 위치, 날짜  

---

## 1.3 전처리 및 가공

- CSV와 API 데이터를 결합하면서 드라이버 및 팀 ID 기준 통일함.  
- 포인트는 float 타입으로 처리하고, 날짜는 YYYY-MM-DD 형식으로 통일함.  

---

## 1.4 데이터베이스 설계

수집한 데이터를 효율적으로 관리하기 위해 드라이버, 팀, 레이스, 서킷 단위로 테이블 구성함.

### 📌 테이블 구성

- **driver_standings**  
  시즌별 드라이버 성적(포인트, 순위, 소속 팀) 저장  

- **drivers_master**  
  드라이버 기본 정보(이름, 생년월일, 국적) 관리  

- **constructors**  
  팀(제조사)의 기본 정보 저장  

- **constructors_standing**  
  팀 성적(포인트, 순위) 저장  

- **races**  
  레이스 일정 및 개최 정보 저장  

- **circuits**  
  서킷 위치 정보 저장  

---

## 🔗 테이블 관계 설계

- **drivers_master ↔ driver_standings**  
  드라이버 기본 정보와 시즌 성적 연결  

- **constructors ↔ driver_standings**  
  드라이버와 소속 팀 연결  

- **constructors ↔ constructors_standing**  
  팀 기본 정보와 성적 연결  

- **circuits ↔ races**  
  서킷과 레이스 정보 연결  

### 1. 드라이버 마스터 (`drivers_master`)

| 컬럼명 | 타입 | 설명 | 비고 |
| :--- | :--- | :--- | :--- |
| **driver_id** | `INT` | 드라이버 고유 식별 번호 | **Primary Key** |
| **first_name** | `VARCHAR(255)` | 이름 | |
| **last_name** | `VARCHAR(255)` | 성 | |
| **birth** | `TIMESTAMP` | 생년월일 | |
| **nationality** | `VARCHAR(255)` | 국적 | |

### 2. 드라이버 시즌 성적 (`driver_standings`)

| 컬럼명 | 타입 | 설명 | 비고 |
| :--- | :--- | :--- | :--- |
| **season_driver_id** | `INT` | 시즌별 드라이버 성적 고유 번호 | **Primary Key** |
| **season** | `INT` | 해당 시즌 | |
| **driver_id** | `INT` | 어느 선수인지 식별 | **Foreign Key** |
| **first_name** | `VARCHAR(255)` | 이름 | |
| **last_name** | `VARCHAR(255)` | 성 | |
| **team_id** | `INT` | 어느 소속인지 식별 | **Foreign Key** |
| **team_name** | `VARCHAR(255)` | 소속 팀 명칭 | |
| **season_point** | `FLOAT` | 해당 시즌 획득 포인트 | |
| **season_positon** | `INT` | 해당 시즌 최종 순위 | |

### 3. 컨스트럭터 성적 (`constructors_standing`)

| 컬럼명 | 타입 | 설명 | 비고 |
| :--- | :--- | :--- | :--- |
| **constructors_standing_id** | `INT` | 컨스트럭터 성적 고유 번호 | **Primary Key** |
| **race_id** | `VARCHAR` | 경기 식별 번호 | |
| **constructors_id** | `INT` | 어느 팀인지 식별 | **Foreign Key** |
| **points** | `FLOAT` | 획득 포인트 | |
| **position** | `INT` | 순위 | |
| **positionText** | `VARCHAR(255)` | 순위 텍스트 정보 | |

### 4. 서킷 정보 (`circuits`)

| 컬럼명 | 타입 | 설명 | 비고 |
| :--- | :--- | :--- | :--- |
| **circuits_id** | `INT` | 서킷 고유 식별 번호 | **Primary Key** |
| **circuitsRef** | `VARCHAR(255)` | 서킷 별칭 | |
| **name** | `VARCHAR(255)` | 서킷 이름 | |
| **location** | `VARCHAR(255)` | 위치 | |
| **country** | `VARCHAR(255)` | 국가 | |
| **lat** | `DOUBLE` | 위도 | |
| **lng** | `DOUBLE` | 경도 | |
| **alt** | `INT` | 고도 | |
| **url** | `VARCHAR(255)` | 관련 웹사이트 URL | |

### 5. 컨스트럭터 정보 (`constructors`)

| 컬럼명 | 타입 | 설명 | 비고 |
| :--- | :--- | :--- | :--- |
| **constructors_id** | `INT` | 컨스트럭터 고유 식별 번호 | **Primary Key** |
| **construtorRef** | `VARCHAR(255)` | 컨스트럭터 별칭 | |
| **name** | `VARCHAR(255)` | 팀 명칭 | |
| **nationality** | `VARCHAR(255)` | 팀 국적 | |
| **url** | `VARCHAR(255)` | 관련 웹사이트 URL | |

### 6. 레이스 일정 정보 (`races`)

| 컬럼명 | 타입 | 설명 | 비고 |
| :--- | :--- | :--- | :--- |
| **race_id** | `INT` | 경기 고유 식별 번호 | **Primary Key** |
| **year** | `INT` | 개최 연도 | |
| **round** | `INT` | 시즌 회차 | |
| **circuit_id** | `INT` | 개최 서킷 식별 번호 | **Foreign Key** |
| **name** | `VARCHAR(255)` | 레이스 명칭 | |
| **date** | `VARCHAR(20)` | 레이스 일자 | |
| **time** | `VARCHAR(20)` | 레이스 시간 | |
| **url** | `VARCHAR(512)` | 관련 정보 URL | |
| **fp1_date** | `VARCHAR(20)` | 주행 일정 | |
| **quali_date** | `VARCHAR(20)` | 예선전 일정 | |
| **sprint_date** | `VARCHAR(20)` | 스프린트 일정 | |